In [ ]:
# @title
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import time

from sklearn import datasets
from IPython.display import clear_output
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# Função imprimir o plano atual
def imprimir_plano(vetores, w, bias, incorretos = None):
    clear_output(wait=True)
    plt.figure()

    plt.xlim(-1, 2)
    plt.ylim(-1, 2)

    y_hiperplano = (((x_hiperplano * -1*w[0]) - bias) / w[1]) # Cuidado, pode dar divisão por zero!
    plt.plot(x_hiperplano, y_hiperplano, color='orange')
    plt.quiver(0, 0, w[0], w[1], color=['b'], angles='xy', scale_units='xy', scale=1)

    for x in vetores:
      if x[2] == -1:
        # Iris Setosa
        plt.plot(x[0], x[1], 'o', color='red')
      else:
        # Iris Versicolour
        plt.plot(x[0], x[1], '+', color='blue')

    if incorretos is not None:
      for x in incorretos:
        plt.plot(x[0], x[1], '2', color='black')

    plt.show()
    return

In [ ]:
# Carga do dataset do repositório do sklearn
iris = datasets.load_iris()

# Colocando no Pandas para filtrar
iris_df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
iris_df["target"] = iris.target

scaler = MinMaxScaler()

# Normalizando características (Necessário para convergir)
iris_df[iris.feature_names] = scaler.fit_transform(iris_df[iris.feature_names])

# Vamos manter apenas as duas primeiras classes: Iris Setosa e Iris Versicolour
vetores_classe_0 = iris_df[iris_df["target"] == 0]
vetores_classe_1 = iris_df[iris_df["target"] == 1]

# Removendo colunas, para deixar o problema bidimensional
remover = ['petal length (cm)', 'petal width (cm)']
vetores_classe_0 = vetores_classe_0.drop(columns = remover)
vetores_classe_1 = vetores_classe_1.drop(columns = remover)

# Ajustando classes para operar com o perceptron
vetores_classe_0["target"] = - 1
vetores_classe_1["target"] = + 1

# Colocando em um vetor numpy para facilitar
vetores = np.concatenate((vetores_classe_0.to_numpy(), vetores_classe_1.to_numpy()))

# Misturando as linhas (Necessário para convergir melhor)
np.random.seed(42)
np.random.shuffle(vetores)

# Valores x da fronteira, apenas para poder visualizar
x_hiperplano = np.array([-1, 2])

# Chute inicial. No mundo real, seria aleatório
w =  np.array([1, -1])
bias = 1
eta = 1

# Implemente aqui o algoritmo de treinamento
max_epocas = 100
print(f"Início do Treinamento: w_inicial={w}, bias_inicial={bias}")

for epoca in range(max_epocas):
    erros = 0
    incorretos = [] # Lista para armazenar pontos incorretos para plotagem
    
    # itera sobre cada ponto (linha) do dataset
    for x_y in vetores:
        # separa o vetor de características e o target
        x = x_y[:2] 
        y = x_y[2]  

        # produto escalar
        a = np.dot(w, x) + bias
        
        # função degrau
        y_predito = 1 if a >= 0 else -1

        # 3. verifica se a predição falhou
        if y != y_predito:
            erros += 1
            incorretos.append(x_y)
            
            # erro positivo
            if y == 1:
                w = w + eta * x
                bias = bias + eta
            
            # erro negativo
            else: # y == -1
                w = w - eta * x
                bias = bias - eta

    imprimir_plano(vetores, w, bias, incorretos)
    
    # se não teve erro, convergiu
    print(f"Época {epoca + 1}/{max_epocas}: Erros = {erros}. w={w}, bias={bias:.4f}")
    
    if erros == 0:
        print(f"\nConvergência alcançada na época {epoca + 1}")
        break
        
else:
    print(f"\nO algoritmo não convergiu após {max_epocas} épocas.")